# 2606-Data Ecosystems and Governance in Organizations

# 0. Load Data

In [42]:
from pymongo import MongoClient
from collections import Counter
import pandas as pd
import numpy as np
import json
import re

with open('data/raw_credit_applications.json') as f:
    data = json.load(f)

print(f"Total records in raw file: {len(data)}")

Total records in raw file: 502


In [43]:
client = MongoClient()
db = client['novacred']
collection = db['applications']
collection.drop()

for record in data:
    collection.replace_one({'_id': record['_id']}, record, upsert=True)

print(f"Records in MongoDB after upsert: {collection.count_documents({})}")
print("(Duplicate IDs were overwritten with their latest version)")

Records in MongoDB after upsert: 500
(Duplicate IDs were overwritten with their latest version)


# 1. Duplicates

In [44]:
# Find duplicates before inserting
ids = [record['_id'] for record in data]
id_counts = Counter(ids)
duplicate_ids = {id_: count for id_, count in id_counts.items() if count > 1}

print(f"Total records in file : {len(data)}")
print(f"Unique _id values     : {len(set(ids))}")
print(f"Duplicate IDs         : {list(duplicate_ids.keys())}")

# Show duplicates
for dup_id in duplicate_ids:
    versions = [r for r in data if r['_id'] == dup_id]
    print(f"\n--- Versions of {dup_id} ---")
    for i, v in enumerate(versions):
        print(f"  Version {i+1}: notes='{v.get('notes', 'none')}', keys={sorted(v.keys())}")

Total records in file : 502
Unique _id values     : 500
Duplicate IDs         : ['app_042', 'app_001']

--- Versions of app_042 ---
  Version 1: notes='none', keys=['_id', 'applicant_info', 'decision', 'financials', 'spending_behavior']
  Version 2: notes='RESUBMISSION', keys=['_id', 'applicant_info', 'decision', 'financials', 'notes', 'spending_behavior']

--- Versions of app_001 ---
  Version 1: notes='none', keys=['_id', 'applicant_info', 'decision', 'financials', 'spending_behavior']
  Version 2: notes='DUPLICATE_ENTRY_ERROR', keys=['_id', 'applicant_info', 'decision', 'financials', 'notes', 'spending_behavior']


**Notes**:

- The raw dataset contained 502 records, of which 500 were unique applications. app_042 was flagged as a resubmission, and app_001 was flagged as a duplicate entry error. For app_042, we'll keep the resubmission as the most recent version to honor the applicant's latest intent.The duplicate entry error of app_001 will be resolved by restoring the original record.

In [45]:
# SSNs as unique IDs. (different IDs, same person?)
ssns = [(r['_id'], r['applicant_info'].get('ssn')) for r in data]
ssn_counts = Counter(ssn for _, ssn in ssns if ssn)
dup_ssns = {ssn: count for ssn, count in ssn_counts.items() if count > 1}

print(f"Duplicate SSNs (same SSN across different app IDs): {len(dup_ssns)}")
if dup_ssns:
    for ssn, count in dup_ssns.items():
        app_ids = [id_ for id_, s in ssns if s == ssn]
        print(f"  SSN {ssn} appears in: {app_ids}")

Duplicate SSNs (same SSN across different app IDs): 3
  SSN 652-70-5530 appears in: ['app_042', 'app_042']
  SSN 937-72-8731 appears in: ['app_101', 'app_234']
  SSN 780-24-9300 appears in: ['app_088', 'app_016']


**Notes**:
- SSN 652-70-5530: Both entries are app_042, so this is just your already-known duplicate ID showing up twice in the raw data. Not a new issue.
- SSN 937-72-8731: Appears in app_101 AND app_234, two different application IDs. Same person, two applications.
- SSN 780-24-9300: Appears in app_088 AND app_016, same situation.

## 1.1 Handle Duplicates

In [47]:
# Fix duplicte entry of app_001, keep correct version
# app_001 — restore original, discard DUPLICATE_ENTRY_ERROR version
original_app_001 = next(r for r in data 
                        if r['_id'] == 'app_001' 
                        and 'DUPLICATE_ENTRY_ERROR' not in str(r.get('notes', '')))

# Change to correct version
collection.replace_one({'_id': 'app_001'}, original_app_001, upsert=True)

print(f"\nFinal document count in MongoDB: {collection.count_documents({})}")


Final document count in MongoDB: 500


In [48]:
# Fix SSN duplicates
unique_data = {r['_id']: r for r in data}
ssns = [(id_, r['applicant_info'].get('ssn')) 
        for id_, r in unique_data.items()]

ssn_counts = Counter(ssn for _, ssn in ssns if ssn)
dup_ssns = {ssn: count for ssn, count in ssn_counts.items() if count > 1}

for ssn, count in dup_ssns.items():
    app_ids = [id_ for id_, s in ssns if s == ssn]
    print(f"  SSN {ssn} appears in: {app_ids}")

# Flag in MongoDB for human review
for ssn in dup_ssns:
    app_ids = [id_ for id_, s in ssns if s == ssn]
    for app_id in app_ids:
        collection.update_one(
            {'_id': app_id},
            {'$set': {'data_quality_flag': 'duplicate_ssn'}}
        )
        print(f"  Flagged {app_id} with 'duplicate_ssn'")

  SSN 937-72-8731 appears in: ['app_101', 'app_234']
  SSN 780-24-9300 appears in: ['app_088', 'app_016']
  Flagged app_101 with 'duplicate_ssn'
  Flagged app_234 with 'duplicate_ssn'
  Flagged app_088 with 'duplicate_ssn'
  Flagged app_016 with 'duplicate_ssn'


The reason you flag rather than delete is important for your README — these are two different application IDs for the same person, so automatically removing one could discard a legitimate credit decision. A human needs to review whether it's a reapplication, fraud, or a data entry error.